# GPT-2 SFT Training Notebook
This notebook clones the repository and runs the full LoRA instruction fine-tuning.

In [ ]:
!git clone https://github.com/amoghsamadhiya779-afk/GPT-PRODUCTION-LEVEL.git
%cd GPT-PRODUCTION-LEVEL
!mkdir -p checkpoints/adapters mlruns data
!pip install -q tiktoken mlflow transformers

### Upload Dataset
Since the dataset isn't in git, you need to upload `sft_mix.jsonl` and `sft_eval.jsonl` into the `data/` directory. You can run this cell to upload them directly.

In [ ]:
from google.colab import files
import os
import shutil

print("Please upload sft_mix.jsonl and sft_eval.jsonl")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, os.path.join("data", filename))
print("Moved dataset files to the data/ directory.")

### 1. The Medium Run (355M)

In [ ]:
# 1. Download official medium weights & map to our architecture
!MODEL_SIZE=medium python training/load_pretrained.py

# 2. Instruction tuning (Automatically runs BEFORE and AFTER eval)
!python training/finetune_instruct.py \
    --model-size medium \
    --device cuda \
    --epochs 2

In [ ]:
!zip -r medium_artifacts.zip checkpoints/adapters/sft_v1_medium.pt mlruns/
from google.colab import files
files.download('medium_artifacts.zip')

### 2. The Small Run Backup (124M)

In [ ]:
# 1. Download official small weights
!MODEL_SIZE=small python training/load_pretrained.py

# 2. Instruction tuning
!python training/finetune_instruct.py \
    --model-size small \
    --device cuda \
    --epochs 2

In [ ]:
!zip -r small_artifacts.zip checkpoints/adapters/sft_v1_small.pt mlruns/
from google.colab import files
files.download('small_artifacts.zip')